# EgoDex Scenario Search on Daft

This notebook reproduces the workflow behind the EgoDex scenario-search blog post with the cleaned `egodex` package. It reads raw EgoDex HDF5 episodes directly, computes hand-pose feature tracks, optionally embeds sampled video frames with SigLIP-2 through Daft, then runs pose-only, text-only, and combined pose plus semantic queries.

The default settings run on one episode so the notebook starts quickly. Switch `DATASET_MODE` to `"full"` when you want blog-scale search across the local dataset.

## 0. Setup

The notebook should run from the repository root or from `datasets/egodex`. It adds `datasets/` to `sys.path` so the local package is importable without an install cell.

In [ ]:
import os
import sys
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "datasets" / "egodex").exists():
            return candidate
    raise RuntimeError("Could not find the daft-examples repository root.")


REPO_ROOT = find_repo_root(Path.cwd().resolve())
os.chdir(REPO_ROOT)
DATASETS_DIR = REPO_ROOT / "datasets"
if str(DATASETS_DIR) not in sys.path:
    sys.path.insert(0, str(DATASETS_DIR))


print(f"repo root: {REPO_ROOT}")

In [ ]:
from pathlib import Path

import daft
from egodex import SCENARIOS, EgoDexPipeline, calibrate, overlay, query
from egodex.features import FPS

DATA_ROOT = Path("datasets/egodex/.data")
if not DATA_ROOT.exists():
    DATA_ROOT = Path(".data")
if not DATA_ROOT.exists():
    raise FileNotFoundError("Set DATA_ROOT to the extracted EgoDex dataset directory.")

DATASET_MODE = "sample"  # "sample" for fast iteration, "full" for blog-scale search
SAMPLE_TASK = "wash_fruit"
SAMPLE_EPISODE_IDS = [0]
TASK_FILTER = None if DATASET_MODE == "full" else [SAMPLE_TASK]
EPISODE_FILTER = None if DATASET_MODE == "full" else SAMPLE_EPISODE_IDS

FEATURES_DIR = Path("datasets/egodex/.features")
EMBEDDINGS_DIR = Path("datasets/egodex/.embeddings")
WRITE_FEATURES = False
WRITE_EMBEDDINGS = False
RUN_EMBEDDINGS = False
RUN_TEXT_QUERIES = False
FRAME_PREVIEW_SECONDS = 5.0

pipeline = EgoDexPipeline(
    str(DATA_ROOT),
    features_dir=str(FEATURES_DIR),
    embeddings_dir=str(EMBEDDINGS_DIR),
)

print(f"Daft {daft.__version__}")
print(f"data root: {DATA_ROOT}")
print(f"task filter: {TASK_FILTER}")
print(f"episode filter: {EPISODE_FILTER}")

## 1. Direct HDF5 Episode Read

`raw()` discovers HDF5 episodes and attaches the sibling MP4 as a lazy video file. The task and episode filters are applied from the path before metadata or trajectory tensors are read, which keeps preview cells cheap.

In [ ]:
episodes = pipeline.raw(tasks=TASK_FILTER, episode_ids=EPISODE_FILTER)
episodes.select("task", "episode_id", "video").show(10)

## 2. Pose Feature Branch

The pose branch stays one row per episode. `trajectory()` reads only the transform tensors needed for pose features, and `calculate_features()` stores each continuous feature as an episode-length track. Query-time scenarios turn those tracks into frame masks and contiguous segments.

In [ ]:
trajectories = pipeline.trajectory(episodes)
features = pipeline.calculate_features(trajectories)
features.select("task", "episode_id", "num_frames").show(10)

In [ ]:
if WRITE_FEATURES:
    features.write_parquet(str(FEATURES_DIR))
    print(f"wrote features to {FEATURES_DIR}")
elif FEATURES_DIR.exists():
    print(f"existing feature parquet available at {FEATURES_DIR}")
else:
    print("feature table is in memory; set WRITE_FEATURES=True to materialize it")

## 3. Scenario Vocabulary

The blog separates physical states from actions. States read a frame's hand shape; actions read motion between frames. Thresholds are calibrated from continuous feature tracks and reused across queries.

In [ ]:
SCENARIO_CATALOG = [
    {"scenario": "openness", "kind": "state", "signal": "closure band"},
    {"scenario": "writing_grip", "kind": "state", "signal": "tripod grip geometry"},
    {"scenario": "hammer_grip", "kind": "state", "signal": "power grip geometry"},
    {"scenario": "twisting", "kind": "action", "signal": "forearm roll rate"},
    {"scenario": "reaching", "kind": "action", "signal": "arm extension rate"},
    {"scenario": "lifting", "kind": "action", "signal": "wrist vertical velocity"},
    {"scenario": "grasping", "kind": "action", "signal": "curl closing rate"},
    {"scenario": "in_hand", "kind": "action", "signal": "still wrist plus finger articulation"},
]

assert set(item["scenario"] for item in SCENARIO_CATALOG) <= set(SCENARIOS)
SCENARIO_CATALOG

In [ ]:
thresholds = calibrate(features)
{key: round(value, 4) for key, value in thresholds.items()}

## 4. Pose-Only Search

Pose-only queries rank episodes by the number of matching frames. Each hit includes exact frame segments, so the result is not just an episode label; it points to where the physical state or action happened.

In [ ]:
POSE_QUERIES = [
    ("open hands", {"pose": "openness", "open_lo": 0.65, "open_hi": 1.0}),
    ("closed hands", {"pose": "openness", "open_lo": 0.0, "open_hi": 0.35}),
    ("writing grip", {"pose": "writing_grip"}),
    ("hammer grip", {"pose": "hammer_grip"}),
    ("twisting", {"pose": "twisting"}),
    ("reaching", {"pose": "reaching"}),
    ("lifting", {"pose": "lifting"}),
    ("grasping", {"pose": "grasping"}),
    ("in-hand manipulation", {"pose": "in_hand"}),
]


def run_pose_queries(k: int = 5) -> dict[str, list[dict[str, object]]]:
    results = {}
    for label, params in POSE_QUERIES:
        results[label] = query(features, k=k, thresholds=thresholds, **params)
    return results


def hit_rows(results: dict[str, list[dict[str, object]]]) -> list[dict[str, object]]:
    rows = []
    for label, hits in results.items():
        if not hits:
            rows.append({"query": label, "task": None, "episode_id": None, "score": 0, "segments": []})
            continue
        for hit in hits:
            rows.append(
                {
                    "query": label,
                    "task": hit["task"],
                    "episode_id": hit["episode_id"],
                    "score": round(float(hit["score"]), 4),
                    "n_frames": hit["n_frames"],
                    "segments": hit["segments"][:3],
                }
            )
    return rows


pose_results = run_pose_queries(k=3)
hit_rows(pose_results)

## 5. Video Frame Sampling

The semantic branch samples video frames at about 1 fps. This preview decodes only the first few seconds so the cell stays bounded.

In [ ]:
preview_frames = pipeline.camera_frames(
    episodes,
    end_time=FRAME_PREVIEW_SECONDS,
    sample_interval_seconds=1.0,
)
preview = preview_frames.select("task", "episode_id", "video_frames").to_pydict()
[
    {
        "task": task,
        "episode_id": episode_id,
        "sampled_frames": len(video_frames),
    }
    for task, episode_id, video_frames in zip(preview["task"], preview["episode_id"], preview["video_frames"])
]

## 6. Optional SigLIP Embeddings

Set `RUN_EMBEDDINGS=True` to run Daft's native `embed_image` expression over sampled frames. First run can download and load the SigLIP model, so it is intentionally opt-in. If `EMBEDDINGS_DIR` already exists, the notebook can load it instead.

In [ ]:
embeddings = None
if RUN_EMBEDDINGS:
    embedding_frames = pipeline.camera_frames(
        episodes,
        sample_interval_seconds=pipeline.sample_interval_seconds,
    )
    embeddings = pipeline.embed_frames(embedding_frames)
    embeddings.select("task", "episode_id", "frame_index", "timestamp", "clip_emb").show(3)
    if WRITE_EMBEDDINGS:
        embeddings.write_parquet(str(EMBEDDINGS_DIR))
        print(f"wrote embeddings to {EMBEDDINGS_DIR}")
elif EMBEDDINGS_DIR.exists():
    embeddings = daft.read_parquet(str(EMBEDDINGS_DIR))
    print(f"loaded embeddings from {EMBEDDINGS_DIR}")
else:
    print("set RUN_EMBEDDINGS=True, or point EMBEDDINGS_DIR at precomputed clip_emb parquet")

## 7. Text and Combined Search

Text-only search ranks sampled frame embeddings by SigLIP similarity and returns a short window around the best frame. Combined search first applies the pose mask, then ranks matching sampled frames by text similarity.

In [ ]:
TEXT_QUERIES = [
    ("text: chopsticks", {"text": "chopsticks"}),
    ("text: stapler", {"text": "stapler"}),
    ("hammer grip + stapler", {"pose": "hammer_grip", "text": "stapler"}),
    ("grasping + shirt", {"pose": "grasping", "text": "shirt"}),
    ("reaching + marbles", {"pose": "reaching", "text": "marbles"}),
]

text_results = {}
if RUN_TEXT_QUERIES and embeddings is not None:
    for label, params in TEXT_QUERIES:
        text_results[label] = query(
            features,
            clip=embeddings,
            k=5,
            thresholds=thresholds,
            **params,
        )
    text_rows = hit_rows(text_results)
else:
    text_rows = [
        {
            "status": "text queries skipped",
            "reason": "set RUN_TEXT_QUERIES=True after embeddings are available",
        }
    ]
text_rows

## 8. Segment Inspection

Use the returned segments to inspect exact frames. This mirrors the blog's dashboard behavior in notebook form: pick a hit, list its segments, then overlay the HDF5 skeleton on frames from the first segment.

In [ ]:
def first_available_hit(
    *result_sets: dict[str, list[dict[str, object]]],
) -> tuple[str | None, dict[str, object] | None]:
    for results in result_sets:
        for label, hits in results.items():
            if hits:
                return label, hits[0]
    return None, None


selected_label, selected_hit = first_available_hit(text_results, pose_results)
selected_label, selected_hit

In [ ]:
if selected_hit is None:
    segment_table = []
else:
    segment_table = [
        {
            "segment": index,
            "start_frame": start,
            "end_frame": end,
            "start_seconds": round(start / FPS, 3),
            "end_seconds": round(end / FPS, 3),
            "duration_seconds": round((end - start + 1) / FPS, 3),
        }
        for index, (start, end) in enumerate(selected_hit["segments"], start=1)
    ]
segment_table

In [ ]:
from IPython.display import display

if selected_hit is not None and selected_hit["segments"]:
    start, end = selected_hit["segments"][0]
    midpoint = (start + end) // 2
    frame_indices = sorted({start, midpoint, end})
    print(f"{selected_label}: {selected_hit['task']}/{selected_hit['episode_id']} segment {start}-{end}")
    for frame_index in frame_indices:
        print(f"frame {frame_index}")
        display(
            overlay(
                DATA_ROOT,
                selected_hit["task"],
                selected_hit["episode_id"],
                frame_index,
                radius=3,
            )
        )
else:
    print("No segment available to visualize.")

## 9. Notes for Blog-Scale Runs

For a full reproduction, set `DATASET_MODE = "full"`, materialize `FEATURES_DIR`, then run embeddings once and write `EMBEDDINGS_DIR`. After those parquets exist, text and combined queries run without recomputing pose features or image embeddings. The notebook keeps inspection inline: use the package API for search and the segment overlay cells for visual review.